Importing all the necessary libraries and packages

In [40]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score

Using os to access the exercise.py file and the dataset file, as well as their respective directories, and using pandas to read the dataset file

In [41]:
notebook_dir = os.getcwd() # Navigating to the notebook directory, which contains the Jupyter Notebook
data_dir = os.path.normpath(os.path.join(notebook_dir, "..", "data")) #Getting the normal path of the data directory

# Using the pandas library, as well as the command read_csv, to read the dataset file
df = pd.read_csv(os.path.join(data_dir, 'WA_Fn-UseC_-Telco-Customer-Churn.csv'))

Cleaning the data by converting 'TotalCharges' column values from string to numericals, replacing missing values with the corresponding median value and removing any duplicates

In [42]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce') # Used to convert TotalCharges from a string to a numeric value
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median()) # Used to replace any missing values in 'TotalCharges' with the median value
df = df.drop_duplicates() # Safely drops any duplicate rows if they exist in the data

columns_to_clean = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in columns_to_clean:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

Performing feature engineering by adding 5 new features

In [43]:
# New Feature #1: Adding a counter functionality for the total number of services a customer uses
services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = (df[services] == 'Yes').sum(axis=1)

# New Feature #2: Calculating the Average Monthly Charge Per Service
df['AvgChargePerService'] = df['MonthlyCharges'] / (df['TotalServices'] + 1)

# New Feature #3: Checks if a customer is a senior with no partner or dependents and separates those citizens
df['IsSeniorAlone'] = ((df['SeniorCitizen'] == 1) & (df['Partner'] == 'No') & (df['Dependents'] == 'No')).astype(int)

# New Feature #4: Checks the Contract duration and converts it to a binary value (0 or 1)
df['LongTermContract'] = df['Contract'].isin(['One year', 'Two year']).astype(int)

# New Feature #5: Checks streaming medium and media
df['UsesStreamingServices'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)

print(f"Shape: {df.shape}")

Shape: (7043, 26)


Splitting and encoding the data

In [44]:
y = df['Churn'].map({'Yes': 1, 'No': 0}).values # Maps 'Yes' and 'No' values in the 'Churn' column to 1 and 0 respectively
X = df.drop(columns=['customerID', 'Churn']) # Drops specific identifiers before numerical vector transformations

# Used to convert categorical variables into numeric columns
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

# Splitting the data rows for training, validating and testing the model
X_train_val, X_test, y_train_val, y_test = train_test_split(X_encoded, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}, Test set: {X_test.shape}")

Training set: (4225, 28), Validation set: (1409, 28), Test set: (1409, 28)


Using StandardScaler to prevent data leakage

In [45]:
scaler = StandardScaler() # Initializing the scaler object 
X_train_scaled = scaler.fit_transform(X_train) # fit_transform is used to train data to prevent data leakage
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# unsqueeze(1) is used to convert vectors from a one-dimensional structure to a two-dimensional structure
train_dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train).unsqueeze(1)) 
val_dataset = TensorDataset(torch.FloatTensor(X_val_scaled), torch.FloatTensor(y_val).unsqueeze(1))
test_dataset = TensorDataset(torch.FloatTensor(X_test_scaled), torch.FloatTensor(y_test).unsqueeze(1))

Calculating Imbalance Target Weight

In [46]:
# Calculating the distribution of retained vs. churned customers in the training split
num_negatives = np.sum(y_train == 0)
num_positives = np.sum(y_train == 1)

# Computing the ratio to penalize missed minor class predictions
imbalance_ratio = num_negatives / num_positives
pos_weight_tensor = torch.FloatTensor([imbalance_ratio])

# Initializing and training the classical baseline model using matching balance configurations
classical_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
classical_model.fit(X_train_scaled, y_train)

# Generating hard predictions and continuous probability scores against the test set
classical_predictions = classical_model.predict(X_test_scaled)
classical_probability = classical_model.predict_proba(X_test_scaled)[:, 1]

print(f"The calculated imbalance target weight is: {imbalance_ratio:.2f}")

The calculated imbalance target weight is: 2.77


Defining the Deep Learning Class Architecture

In [47]:
class ChurnMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, dropout_rate=0.2):
        super(ChurnMLP, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim1) # Accepts input dimension maps and scales them to hidden_dim1
        self.act1 = nn.ReLU() # ReLU is used for learning non-linear shapes
        self.dropout1 = nn.Dropout(dropout_rate) # Convets weights to 0 to prevent feature co-adaptation
        
        self.layer2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.act2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.output_layer = nn.Linear(hidden_dim2, 1)
        
    def forward(self, x):
        x = self.dropout1(self.act1(self.layer1(x)))
        x = self.dropout2(self.act2(self.layer2(x)))
        return self.output_layer(x)

Optimizing the model and the recursive training loop

In [48]:
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 100
HIDDEN_1 = 64
HIDDEN_2 = 32
DROPOUT = 0.2

# Initializing the DataLoader to shuffle and batch our scaled training partitions smoothly
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Instantiating the model class, pulling the feature count dimension directly from the dataset shape
model = ChurnMLP(X_train_scaled.shape[1], HIDDEN_1, HIDDEN_2, dropout_rate=DROPOUT)

# Using BCEWithLogitsLoss with pos_weight ensures the minor churn class is heavily weighted
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # Adam adjusts unique learning step limits for the node weights

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train() # Explicitly switches on dropout behaviors to stabilize network updates
    running_train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()                  # Clearing structural gradient remnants from previous step
        predictions = model(batch_X)           # Forward pass: Feed features through layer weights
        loss = criterion(predictions, batch_y) # Computing error penalty based on target label mismatch
        loss.backward()                        # Computing partial derivative slopes
        optimizer.step()                       # Updating weight balances based on optimization math
        running_train_loss += loss.item() * batch_X.size(0)
        
    model.eval() # Removes structural dropout layers
    with torch.no_grad(): # Deactivates graph tracking history
        val_X, val_y = val_dataset.tensors
        val_preds = model(val_X)
        val_loss = criterion(val_preds, val_y)
        running_val_loss = val_loss.item() * val_X.size(0)
        
    # Calculating and storing the normalized average epoch values for plotting loss metrics later
    epoch_train_loss = running_train_loss / len(train_dataset)
    epoch_val_loss = running_val_loss / len(val_dataset)
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

Performing the final evaluation and comparison

In [49]:
model.eval()
with torch.no_grad():
    test_X, test_y = test_dataset.tensors
    raw_test_logits = model(test_X)
    nn_probs = torch.sigmoid(raw_test_logits).numpy().flatten()
    nn_preds = (nn_probs >= 0.50).astype(int)

print(f"Classical Logistic Regression Test Accuracy: {accuracy_score(y_test, classical_predictions):.2%}")
print(f"Deep Learning Neural Network Test Accuracy:  {accuracy_score(y_test, nn_preds):.2%}")

print(confusion_matrix(y_test, classical_predictions))
print(confusion_matrix(y_test, nn_preds))

print(classification_report(y_test, classical_predictions))
print(classification_report(y_test, nn_preds))

print(f"Classical Model ROC-AUC Score:      {roc_auc_score(y_test, classical_predictions):.4f}")
print(f"Neural Network Model ROC-AUC Score: {roc_auc_score(y_test, nn_probs):.4f}")


Classical Logistic Regression Test Accuracy: 74.24%
Deep Learning Neural Network Test Accuracy:  74.95%
[[754 281]
 [ 82 292]]
[[756 279]
 [ 74 300]]
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.78      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.75      0.71      1409
weighted avg       0.80      0.74      0.76      1409

              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.52      0.80      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409

Classical Model ROC-AUC Score:      0.7546
Neural Network Model ROC-AUC Score: 0.8329
